# DRGU Mini-Batch Implementation

This notebook demonstrates the mini-batch Fisher scoring implementation of DRGU for scalable training.


In [1]:
import numpy as np
import pandas as pd

# Import RobustInfer components
from robustinfer import DRGU, DRGUMiniBatch


## Generate Simulation Dataset

We'll create a dataset to demonstrate mini-batch training benefits.


In [2]:
# Generate synthetic dataset
np.random.seed(42)
n_obs = 2000
p_features = 3

# Generate covariates
X = np.random.normal(0, 1, (n_obs, p_features))

# Generate treatment (binary)
treatment_prob = 1 / (1 + np.exp(-(X @ np.array([0.5, -0.3, 0.2]))))
treatment = np.random.binomial(1, treatment_prob)

# Generate response with treatment effect
true_effect = 0.5  
y = (X @ np.array([0.2, 0.3, -0.1]) + 
     true_effect * treatment + 
     np.random.normal(0, 0.5, n_obs))

# Create DataFrame
df = pd.DataFrame({
    'response': y,
    'treatment': treatment,
    'x1': X[:, 0],
    'x2': X[:, 1], 
    'x3': X[:, 2]
})

print(f"Dataset: {n_obs} observations, {p_features} features")
print(f"Treatment prevalence: {treatment.mean():.3f}")
print(f"True treatment effect: {true_effect}")
df.head()


Dataset: 2000 observations, 3 features
Treatment prevalence: 0.514
True treatment effect: 0.5


,response,treatment,x1,x2,x3
0,0.824363,1,0.496714,-0.138264,0.647689
1,0.346346,1,1.523030,-0.234153,-0.234137
2,0.573410,1,1.579213,0.767435,-0.469474
3,0.925050,1,0.542560,-0.463418,-0.465730
4,-0.154916,0,0.241962,-1.913280,-1.724918


## Compare Full-Batch vs Mini-Batch DRGU

Now let's compare the original full-batch DRGU with the mini-batch implementation.


In [3]:
# Full-batch DRGU
model_full = DRGU(df, 
                  covariates=["x1", "x2", "x3"], 
                  treatment="treatment", 
                  response="response")
model_full.fit()
print("Full-batch DRGU fitted")
model_full.summary()


Step 0 gradient norm: 0.107547
Converged after 6 iterations
Full-batch DRGU fitted


,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.719837,0.5,0.010738,20.472517,0.000000e+00
1,beta_0,0.051409,0.0,0.047477,1.082812,2.788918e-01
2,beta_1,0.628461,0.0,0.052749,11.914235,0.000000e+00
3,beta_2,-0.242979,0.0,0.047122,-5.156421,2.517150e-07
4,beta_3,0.304192,0.0,0.048862,6.225529,4.799348e-10
5,gamma_0,1.233293,0.0,0.070925,17.388716,0.000000e+00
6,gamma_1,0.402317,0.0,0.060881,6.608307,3.887402e-11
7,gamma_2,0.727092,0.0,0.058033,12.529000,0.000000e+00
8,gamma_3,-0.296891,0.0,0.051104,-5.809520,6.265212e-09
9,gamma_4,-0.462010,0.0,0.054755,-8.437843,0.000000e+00


In [3]:
# Mini-batch DRGU
model_mini = DRGUMiniBatch(df,
                          covariates=["x1", "x2", "x3"],
                          treatment="treatment",
                          response="response")

# Fit with mini-batch parameters
model_mini.fit(max_step_norm=30, tol = 0.5, verbose = True)


DRGU: n=2000, epochs=10, tol=5.0e-01, batches_per_epoch=20
Epoch 1/10
  Step 1: delta=3.079e-01, u=1.896e-01
  CONVERGED at step 1
CONVERGED after 1 steps (final delta=3.079e-01)
  Final u_norm: 1.896e-01


{'delta': tensor([0.6649]),
 'beta': tensor([-0.0539,  0.5101, -0.2526,  0.3803]),
 'gamma': tensor([ 0.7631, -0.0647, -0.0402, -0.1386, -0.0810, -0.0303, -0.1545]),
 'converged': True,
 'iterations': 1,
 'final_delta_norm': 0.3078711214473563,
 'epochs_run': 1}

In [4]:
model_mini.estimate_variance(pairs_per_anchor = 99, s=500)

Variance estimation: s=500, k=99
MC estimation: n=2000, s=500, m=99
  Batch 100
  Batch 200
  Batch 300
  Batch 400
  Batch 500
Anchor-based variance computed
Variance matrix computed


In [5]:
model_mini.summary()

,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.664891,0.5,0.023405,7.045223,1.851630e-12
1,beta_0,-0.053906,0.0,0.048003,-1.122973,2.614491e-01
2,beta_1,0.510059,0.0,0.052656,9.686623,0.000000e+00
3,beta_2,-0.252584,0.0,0.048119,-5.249147,1.528052e-07
4,beta_3,0.380343,0.0,0.050072,7.595967,3.064216e-14
5,gamma_0,0.763117,0.0,0.065507,11.649472,0.000000e+00
6,gamma_1,-0.064725,0.0,0.067126,-0.964245,3.349232e-01
7,gamma_2,-0.040247,0.0,0.080758,-0.498369,6.182239e-01
8,gamma_3,-0.138579,0.0,0.062657,-2.211718,2.698615e-02
9,gamma_4,-0.080994,0.0,0.067611,-1.197933,2.309431e-01
